# Chapter 8 — Linking Ontologies to Data
### Notebook 4 · Agentic lab — mapping shapes and execution strategy

*Book reference: Extends §8.2–8.3*

An agent that makes the two decisions of this chapter, and an MDP in which the cheapest action is also the one that returns wrong answers.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch08_toolkit as ch8
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [3]:
import ch08_agentic as AG
from oe_course import evaluation as ev, llm, mdp, optimize as opt
import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


**By the end of this notebook you can:**

1. Give an agent a tool that checks two execution paths agree, and see why that beats any prompt instruction.
2. Stratify a dataset on **two independent dimensions** at once.
3. Model a workload as an MDP where staleness is lost **reward**, not added latency.
4. Read an optimal refresh policy and explain each decision economically.

> **Prerequisite:** the Chapter 1 agentic lab.

## 1. Tools

`compare_strategies` is the one that matters. An agent that calls it cannot ship the IRI/literal bug, regardless of what it believes about mappings — the distinction is enforced by a tool rather than asserted in a prompt.

In [4]:
ctx = AG.Ch8Context()
tools = {t.name: t for t in AG.build_toolset(ctx)}
for name, t in tools.items():
    print(f'{name:22s} {list(t.args_schema.model_json_schema().get("properties", {}))}')
    print(f'{"":22s} {t.description.splitlines()[0]}')

inspect_schema         []
                       List the source tables, their columns and their foreign keys.
show_mappings          []
                       Show the current mappings and what each one produces.
materialise_now        []
                       Run every mapping and report the size of the resulting graph.
run_query              ['name', 'strategy']
                       Answer one of the chapter's named queries by 'rewrite' or 'materialise'.
compare_strategies     ['name']
                       Check that materialisation and rewriting agree on a query.
show_sql               ['name']
                       Show the SQL a query rewrites to — useful for explaining a plan.


In [5]:
schema = json.loads(tools['inspect_schema'].invoke({}))
print('patient table:')
print('  columns     :', [c['name'] for c in schema['patient']['columns']])
print('  foreign keys:', schema['patient']['foreign_keys'])
print()
print(tools['compare_strategies'].invoke({'name': 'cardiac-patients-in-cardiology'}))
print(tools['run_query'].invoke({'name': 'patients-in-cardiology',
                                 'strategy': 'rewrite'}))
print('\ntrajectory:', ctx.log.names())

patient table:
  columns     : ['id', 'name', 'ward_id']
  foreign keys: [{'column': 'ward_id', 'references': 'ward.id'}]



{"agree": true, "materialised": 3, "rewritten": 3}
{"strategy": "rewrite", "rows": 3, "answers": [{"p": "http://example.org/data/patient/101", "w": "http://example.org/data/ward/1"}, {"p": "http://example.org/data/patient/103", "w": "http://example.org/data/ward/3"}, {"p": "http://example.org/data/patient/104", "w": "http://example.org/data/ward/1"}]}

trajectory: ['inspect_schema', 'compare_strategies', 'run_query']


## 2. The dataset, stratified on two dimensions

Each case pairs a **column** (foreign key or attribute) with a **workload** (static or volatile), and the agent must get both right. Those are independent, so the split has to stratify on both — alternating over the raw list would have put every static workload in train and every volatile one in dev, leaving each half unable to teach one of the strategies.

In [6]:
train, dev = AG.build_dataset('train'), AG.build_dataset('dev')
print(pd.DataFrame([{'id': e.id, 'object_kind': e.gold_object_kind,
                     'strategy': e.gold_strategy,
                     'split': 'train' if e in train else 'dev'}
                    for e in AG.build_dataset('all')]).to_string(index=False))
print()
print('train combinations:', sorted({(e.gold_object_kind, e.gold_strategy) for e in train}))
print('dev   combinations:', sorted({(e.gold_object_kind, e.gold_strategy) for e in dev}))
assert ({(e.gold_object_kind, e.gold_strategy) for e in train}
        == {(e.gold_object_kind, e.gold_strategy) for e in dev})

                id object_kind    strategy split
     inward-static         iri materialise train
       inward-live         iri     rewrite train
 speciality-static     literal materialise train
   speciality-live     literal     rewrite train
hasdisorder-static         iri materialise   dev
  hasdisorder-live         iri     rewrite   dev
   category-static     literal materialise   dev
     category-live     literal     rewrite   dev
      label-static     literal materialise train
      wardref-live         iri     rewrite train

train combinations: [('iri', 'materialise'), ('iri', 'rewrite'), ('literal', 'materialise'), ('literal', 'rewrite')]
dev   combinations: [('iri', 'materialise'), ('iri', 'rewrite'), ('literal', 'materialise'), ('literal', 'rewrite')]


## 3. Baseline and GEPA

The naive agent does what real projects do: emits literals unless told otherwise, and loads everything into a warehouse.

In [7]:
lm = llm.configure_dspy(AG.OBDA_RULEBOOK, AG.obda_responder)
baseline = AG.OBDAProgram()
for e in dev:
    p = baseline(**e.inputs())
    print(f'{e.id:20s} answered {p.object_kind:8s}/{p.strategy:12s} '
          f'gold {e.gold_object_kind:8s}/{e.gold_strategy}')

hasdisorder-static   answered literal /materialise  gold iri     /materialise
hasdisorder-live     answered literal /materialise  gold iri     /rewrite
category-static      answered literal /materialise  gold literal /materialise
category-live        answered literal /materialise  gold literal /rewrite


In [8]:
before = ev.evaluate_dataset(baseline, dev, AG.obda_scorer)
print('BEFORE:', before['mean_score'], before['violations'])

BEFORE: 0.5 {'iri-for-foreign-keys': 2, 'rewrite-for-volatile-data': 2}


In [9]:
gepa_metric = ev.make_gepa_metric(AG.obda_scorer, AG.OBDA_RULEBOOK)
reflect = llm.reflection_lm(AG.OBDA_RULEBOOK, AG.obda_responder)
tuned = opt.run_gepa(baseline, train, gepa_metric, valset=train,
                     max_metric_calls=80, reflection_lm=reflect)
result = opt.compare(AG.OBDAProgram(), tuned, dev, AG.obda_scorer)
print(result.report())

2026/08/24 18:50:41 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 80 metric calls of the program. This amounts to 6.67 full evals on the train+val set.


2026/08/24 18:50:41 INFO dspy.teleprompt.gepa.gepa: Using 6 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


GEPA Optimization:   0%|          | 0/80 [00:00<?, ?rollouts/s]

2026/08/24 18:50:41 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 6 (50.0%)


2026/08/24 18:50:41 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.5


...

2026/08/24 18:50:42 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 2 (100.0%)


2026/08/24 18:50:42 INFO dspy.teleprompt.gepa.gepa: Iteration 33: All subsample scores perfect. Skipping.


2026/08/24 18:50:42 INFO dspy.teleprompt.gepa.gepa: Iteration 33: Reflective mutation did not propose a new candidate


GEPA Optimization:  98%|█████████▊| 78/80 [00:01<00:00, 72.56rollouts/s]

mean score  0.500  ->  1.000   (delta +0.500)
violations  {'iri-for-foreign-keys': 2, 'rewrite-for-volatile-data': 2}
        ->  {}

instruction diff:
--- instruction (before)
+++ instruction (after)
@@ -1 +1,3 @@
 You are an OBDA engineer. Decide how to map the column and how to run the queries.
+- RULE iri-for-foreign-keys: When the object column is a foreign key referencing another table, map it to an IRI built from that table's template (a referencing object map). A literal there disconnects the graph silently.
+- RULE rewrite-for-volatile-data: When the source changes continuously and answers must be current, use query rewriting: a materialised copy would serve stale answers.


In [10]:
found = AG.OBDA_RULEBOOK.active_in(result.instruction_after)
print('rules discovered:', sorted(found))
print('rules not needed:', sorted(set(AG.OBDA_RULEBOOK.ids) - found))
print('\nThe two undiscovered rules describe the agent\'s DEFAULT behaviour --\n'
      'literals for attributes, materialise for static data. It was already\n'
      'doing those, so the metric never complained and there was nothing to\n'
      'learn. An undiscovered rule is not automatically a failure; check\n'
      'whether it was ever violated before concluding anything.')

rules discovered: ['iri-for-foreign-keys', 'rewrite-for-volatile-data']
rules not needed: ['literal-for-attributes', 'materialise-for-read-heavy']

The two undiscovered rules describe the agent's DEFAULT behaviour --
literals for attributes, materialise for static data. It was already
doing those, so the metric never complained and there was nothing to
learn. An undiscovered rule is not automatically a failure; check
whether it was ever violated before concluding anything.


## 4. The materialise-or-rewrite MDP

A known workload interleaves queries (`q`) with updates (`u`). An update makes any materialised copy stale.

| | |
|---|---|
| **S** | position in the workload, and whether the copy is fresh |
| **A** | serve by rewriting · serve from the copy · refresh, then serve |
| **T** | deterministic |
| **R** | `+1` for a **correct** answer, minus the cost of serving it |

The trap is deliberate. Serving from a stale copy is the **cheapest** action and earns **nothing**, because the answer is wrong. An agent optimising latency alone walks straight into it.

In [11]:
M = AG.MaterialisationMDP(workload='qquqqquq')
print('workload:', M.workload, ' (q = query, u = update)')
print(f'costs: rewrite={M.cost_rewrite}, from copy={M.cost_materialised}, '
      f'refresh={M.cost_refresh}')
V, pi = mdp.value_iteration(M)
s0 = M.initial_state()
print(f'\nV*(s0) = {V[s0]:.3f}   (8 events, 6 of them queries)')

workload: qquqqquq  (q = query, u = update)
costs: rewrite=0.3, from copy=0.05, refresh=0.5

V*(s0) = 4.450   (8 events, 6 of them queries)


In [12]:
state = s0
while not M.is_terminal(state):
    action = pi[state]
    print('  ' + M.describe(state, action))
    state = M.transition(state, action)[0][1]

  [0] query  fresh=False -> serve:rewrite
  [1] query  fresh=False -> serve:rewrite
  [2] update fresh=False -> apply-update
  [3] query  fresh=False -> serve:refresh-first
  [4] query  fresh=True  -> serve:materialised
  [5] query  fresh=True  -> serve:materialised
  [6] update fresh=True  -> apply-update
  [7] query  fresh=False -> serve:rewrite


> **Read the policy as economics.** The first two queries are served by rewriting — an update is coming, so a refresh would be wasted. After the update there are *three* queries before the next one, so the agent pays for a refresh once and serves the rest cheaply from the copy. After the final update only one query remains, so it rewrites again rather than refreshing.

Nobody encoded that rule. It falls out of the costs.

In [13]:
policies = {
    'always rewrite': lambda s, acts: 'apply-update' if 'apply-update' in acts else 'serve:rewrite',
    'always from copy': lambda s, acts: 'apply-update' if 'apply-update' in acts else 'serve:materialised',
    'refresh every query': lambda s, acts: 'apply-update' if 'apply-update' in acts else 'serve:refresh-first',
    'optimal': mdp.greedy_policy(pi),
}
rows = []
for name, policy in policies.items():
    ep = mdp.run_episode(M, policy, max_steps=40)
    rows.append({'policy': name, 'return': round(ep.discounted_return(), 3)})
print(pd.DataFrame(rows).to_string(index=False))
print('\n"Always from copy" is the cheapest and by far the worst: it answers\n'
      'most queries from stale data and earns nothing for them.')

             policy  return
     always rewrite    4.20
   always from copy   -0.30
refresh every query    2.70
            optimal    4.45

"Always from copy" is the cheapest and by far the worst: it answers
most queries from stale data and earns nothing for them.


### Exercise 4.1 — Find the update rate that flips the strategy

Sweep workloads from update-heavy to read-heavy and report where the optimal policy stops refreshing at all, and where it starts serving mostly from the copy.

In [14]:
# YOUR CODE HERE


<details>
<summary>Solution 4.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [15]:
rows = []
for workload in ['ququququ', 'qquqquq', 'qqquqqqu', 'qqqqqqqu', 'qqqqqqqq']:
    Mw = AG.MaterialisationMDP(workload=workload)
    Vw, piw = mdp.value_iteration(Mw)
    state, actions = Mw.initial_state(), []
    while not Mw.is_terminal(state):
        a = piw[state]; actions.append(a)
        state = Mw.transition(state, a)[0][1]
    rows.append({'workload': workload,
                 'updates': workload.count('u'),
                 'V*': round(Vw[Mw.initial_state()], 3),
                 'refreshes': actions.count('serve:refresh-first'),
                 'from copy': actions.count('serve:materialised'),
                 'rewrites': actions.count('serve:rewrite')})
print(pd.DataFrame(rows).to_string(index=False))
print('\nAs updates get rarer the policy shifts from pure rewriting to refresh-\n'
      'then-reuse. The crossover is not a rule of thumb -- it is the point where\n'
      'the refresh cost is amortised over enough queries, and it moves as soon\n'
      'as any of the three costs changes.')

workload  updates   V*  refreshes  from copy  rewrites
ququququ        4 2.80          0          0         4
 qquqquq        2 3.50          1          1         3
qqquqqqu        2 4.70          2          4         0
qqqqqqqu        1 6.15          1          6         0
qqqqqqqq        0 7.10          1          7         0

As updates get rarer the policy shifts from pure rewriting to refresh-
then-reuse. The crossover is not a rule of thumb -- it is the point where
the refresh cost is amortised over enough queries, and it moves as soon
as any of the three costs changes.


### Exercise 4.2 — Remove the staleness penalty and watch the agent go wrong

Change the reward so a stale answer still scores `+1` — i.e. model latency only. Show the optimal policy becoming one that mostly serves wrong answers.

In [16]:
# YOUR CODE HERE


<details>
<summary>Solution 4.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [17]:
class LatencyOnlyMDP(AG.MaterialisationMDP):
    """Rewards speed and ignores correctness -- a common dashboard SLA."""
    def transition(self, state, action):
        nxt = AG.ServeState(state.index + 1, state.fresh)
        if action == 'apply-update':
            return [(1.0, AG.ServeState(state.index + 1, False), 0.0)]
        if action == 'serve:rewrite':
            return [(1.0, nxt, 1.0 - self.cost_rewrite)]
        if action == 'serve:materialised':
            return [(1.0, nxt, 1.0 - self.cost_materialised)]   # no staleness penalty
        return [(1.0, AG.ServeState(state.index + 1, True),
                 1.0 - self.cost_refresh - self.cost_materialised)]

M2 = LatencyOnlyMDP(workload='qquqqquq')
V2, pi2 = mdp.value_iteration(M2)
state, actions, stale_answers = M2.initial_state(), [], 0
while not M2.is_terminal(state):
    a = pi2[state]
    if a == 'serve:materialised' and not state.fresh:
        stale_answers += 1
    actions.append(a); state = M2.transition(state, a)[0][1]
print('latency-only optimal policy:')
for a in actions: print('   ', a)
print(f'\nanswers served from STALE data: {stale_answers}')
assert stale_answers >= 4
print('\nThe policy is optimal for the reward it was given, and it serves wrong\n'
      'answers to most queries. This is the whole argument for putting\n'
      'correctness in the reward: an agent will exploit any gap between what\n'
      'you measure and what you meant, and it will do so efficiently.')

latency-only optimal policy:
    serve:materialised
    serve:materialised
    apply-update
    serve:materialised
    serve:materialised
    serve:materialised
    apply-update
    serve:materialised

answers served from STALE data: 6

The policy is optimal for the reward it was given, and it serves wrong
answers to most queries. This is the whole argument for putting
correctness in the reward: an agent will exploit any gap between what
you measure and what you meant, and it will do so efficiently.


### Exercise 4.3 — Give the agent a mapping it must verify

Use `compare_strategies` to show that a broken mapping is caught without any gold answer, and argue why this belongs in a tool rather than in the prompt.

In [18]:
# YOUR CODE HERE


<details>
<summary>Solution 4.3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [19]:
ctx2 = AG.Ch8Context()
t2 = {t.name: t for t in AG.build_toolset(ctx2)}
print('with the correct mappings:')
for name in ch8.QUERIES:
    print('  ', name, t2['compare_strategies'].invoke({'name': name}))

# Now break one mapping and re-check by hand.
broken = {'classes': ch8.MAPPINGS['classes'],
          'properties': [ch8.PropertyMap(p.predicate, p.table, p.subject_column,
                                         p.object_column, p.subject_template,
                                         None if p.predicate.endswith('inWard')
                                         else p.object_template)
                         for p in ch8.MAPPINGS['properties']]}
q = ch8.QUERIES['patients-in-cardiology']
left = ch8.answers_via_materialisation(ctx2.conn, q, mappings=broken)
right = ch8.answers_via_rewriting(ctx2.conn, q, mappings=broken)
print('\nwith a foreign key mapped to a literal:')
print('  materialised', len(left), 'rows; rewritten', len(right), 'rows;',
      'agree:', left == right)
assert left != right
print('\nA prompt saying "remember to use IRIs for foreign keys" is advice, and\n'
      'advice is followed unevenly. A tool that compares two execution paths is\n'
      'a CHECK: it fails loudly on exactly this bug, needs no gold answer, and\n'
      'keeps working after the prompt is rewritten or the model is swapped.')

with the correct mappings:
   patients {"agree": true, "materialised": 5, "rewritten": 5}
   patients-in-cardiology {"agree": true, "materialised": 3, "rewritten": 3}
   patients-with-cardiac-disorder {"agree": true, "materialised": 3, "rewritten": 3}


   cardiac-patients-in-cardiology {"agree": true, "materialised": 3, "rewritten": 3}

with a foreign key mapped to a literal:
  materialised 0 rows; rewritten 3 rows; agree: False

A prompt saying "remember to use IRIs for foreign keys" is advice, and
advice is followed unevenly. A tool that compares two execution paths is
a CHECK: it fails loudly on exactly this bug, needs no gold answer, and
keeps working after the prompt is rewritten or the model is swapped.


## Chapter 8 in the course arc

| | Ch. 5 | Ch. 6 | Ch. 8 |
|---|---|---|---|
| MDP | plan under prerequisites | diagnosis | **serve under staleness** |
| the error it prevents | ontologically wrong axiom | unsound part-whole chaining | **confident stale answers** |
| how it is caught | meta-property checker | relation taxonomy | **two execution paths disagreeing** |

Chapter 8's contribution to the course's argument about evaluation: the best checks need no gold answer. Running the same question two ways and demanding agreement catches a whole class of bug for free — and it is the pattern to reach for whenever you have two independent routes to the same result.